# NutriSmart — Knowledge Graph (Neo4j Desktop, Ontology-Aligned)

**Database:** Neo4j Desktop · `bolt://localhost:7687`  
**Schema version:** ontology-aligned (`HAS_INGREDIENT`, `BELONGS_TO_CUISINE`, `HAS_NUTRITION`, `Cuisine`, `NutritionalProfile`)

This notebook:
1. Runs `ingest_desktop.py` to populate the Neo4j Desktop instance with the full ontology schema
2. Verifies node/edge counts and schema labels
3. Executes RQ-aligned Cypher queries and exports results

**Prerequisites**
- Neo4j Desktop running with database `nutrismart` started
- Python driver installed: `pip install neo4j pandas scipy`


In [1]:
## Cell 1 — Run ingestion script (ingest_desktop.py)
import subprocess, sys, os

# NOTE: Adjust the path to your ingest_desktop.py script as needed
result = subprocess.run([sys.executable, "ingest_desktop.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


Connecting to Neo4j Desktop at bolt://localhost:7687 ...
Connected to Neo4j Desktop.

Database cleared.
Constraints and indexes created.

Ingesting data...
  Loaded 9,997 rows via _load_recipe_batch
  Loaded 58 rows via _load_ingredient_batch
  Loaded 103,908 rows via _load_has_ingredient_batch

Applying inference rules...
  Inference R1 applied: 9,997 recipes updated with total_co2e and carbon_tier

Creating NutritionalProfile nodes...
  NutritionalProfile nodes created: 9,997

=== Graph verification ===
  Recipe nodes                        9,997
  Ingredient nodes                       58
  Cuisine nodes                           5
  NutritionalProfile nodes            9,997
  HAS_INGREDIENT edges               96,139
  BELONGS_TO_CUISINE edges            9,997
  HAS_NUTRITION edges                 9,997
  Recipes w/ carbon_tier              9,997
  Recipes w/ total_co2e               9,997
  HAS_INGREDIENT w/ co2e             96,139

  Node labels:         ['Cuisine', 'Ingredient',

In [2]:
## Cell 2 — Connect driver (reused across all query cells)
# Target: Neo4j Desktop — password set when creating the database in Desktop app
from neo4j import GraphDatabase
import pandas as pd

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "nutrismart12311")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connected to Neo4j Desktop.")


Connected to Neo4j Desktop.


In [3]:
## Cell 3 — Verification: node and edge counts + schema labels

def run_query(cypher):
    with driver.session() as s:
        return s.run(cypher).data()

checks = {
    "Recipe nodes":             "MATCH (r:Recipe)               RETURN count(r) AS n",
    "Ingredient nodes":         "MATCH (i:Ingredient)           RETURN count(i) AS n",
    "Cuisine nodes":            "MATCH (c:Cuisine)              RETURN count(c) AS n",
    "NutritionalProfile nodes": "MATCH (np:NutritionalProfile)  RETURN count(np) AS n",
    "HAS_INGREDIENT edges":     "MATCH ()-[h:HAS_INGREDIENT]->()     RETURN count(h) AS n",
    "BELONGS_TO_CUISINE edges": "MATCH ()-[b:BELONGS_TO_CUISINE]->() RETURN count(b) AS n",
    "HAS_NUTRITION edges":      "MATCH ()-[n:HAS_NUTRITION]->()      RETURN count(n) AS n",
    "Recipes w/ carbon_tier":   "MATCH (r:Recipe) WHERE r.carbon_tier IS NOT NULL RETURN count(r) AS n",
    "Recipes w/ total_co2e":    "MATCH (r:Recipe) WHERE r.total_co2e  IS NOT NULL RETURN count(r) AS n",
    "HAS_INGREDIENT w/ co2e":   "MATCH ()-[h:HAS_INGREDIENT]->() WHERE h.co2e_kg IS NOT NULL RETURN count(h) AS n",
}

print("=== Graph statistics ===")
for label, q in checks.items():
    n = run_query(q)[0]["n"]
    print(f"  {label:<30} {n:>10,}")

labels = [r["label"] for r in run_query("CALL db.labels() YIELD label RETURN label ORDER BY label")]
rels   = [r["relationshipType"] for r in run_query("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType")]
print(f"\n  Node labels:        {labels}")
print(f"  Relationship types: {rels}")

expected_labels = {"Cuisine", "Ingredient", "NutritionalProfile", "Recipe"}
expected_rels   = {"BELONGS_TO_CUISINE", "HAS_INGREDIENT", "HAS_NUTRITION"}
missing_l = expected_labels - set(labels)
missing_r = expected_rels - set(rels)
if missing_l or missing_r:
    print(f"\n  WARNING — missing labels: {missing_l}, missing rels: {missing_r}")
else:
    print("\n  Graph matches ontology schema. ✓")


=== Graph statistics ===
  Recipe nodes                        9,997
  Ingredient nodes                       58
  Cuisine nodes                           5
  NutritionalProfile nodes            9,997
  HAS_INGREDIENT edges               96,139
  BELONGS_TO_CUISINE edges            9,997
  HAS_NUTRITION edges                 9,997
  Recipes w/ carbon_tier              9,997
  Recipes w/ total_co2e               9,997
  HAS_INGREDIENT w/ co2e             96,139

  Node labels:        ['Cuisine', 'Ingredient', 'NutritionalProfile', 'Recipe']
  Relationship types: ['BELONGS_TO_CUISINE', 'HAS_INGREDIENT', 'HAS_NUTRITION']

  Graph matches ontology schema. ✓


In [4]:
## Cell 4 — Sample: top 10 ingredient CO₂ contributions across the graph
sample_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
RETURN r.name               AS recipe,
       i.ingredient_name    AS ingredient,
       round(h.grams, 1)    AS grams,
       round(h.co2e_kg, 4)  AS co2e_kg
ORDER BY h.co2e_kg DESC
LIMIT 10
"""
sample_df = pd.DataFrame(run_query(sample_q))
print("Top 10 ingredient CO₂ contributions across the graph:")
display(sample_df)


Top 10 ingredient CO₂ contributions across the graph:


,recipe,ingredient,grams,co2e_kg
0,Shorshe Ilish,beef,999.0,59.7902
1,Indian Desserts Special 351,beef,999.0,59.7902
2,Hakka Noodles,beef,999.0,59.7902
3,Bangladeshi Desserts Special 15,beef,999.0,59.7902
4,Fusion Street Food Special 240,beef,999.0,59.7902
5,Nihari,beef,998.0,59.7303
6,Mantu,beef,998.0,59.7303
7,Chapli Kabab,beef,997.0,59.6705
8,Haleem,beef,996.0,59.6106
9,Nihari,beef,995.0,59.5508


## RQ1 — Which ingredients drive the highest recipe CO₂, and how often do they appear in top-rated recipes?

Two sub-queries:
- **RQ1a**: Ingredients with the highest average CO₂ contribution per recipe they appear in (dominant ingredient per recipe)
- **RQ1b**: Among the top-1000 highest-rated recipes, which ingredients appear most frequently, and what is their average CO₂ contribution?


In [5]:
## RQ1a — Dominant CO₂ ingredient per recipe, ranked by avg contribution
rq1a_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
WITH r, i, h.co2e_kg AS contrib
ORDER BY contrib DESC
WITH r, collect({ingredient: i.ingredient_name, co2: contrib})[0] AS top_ingr
WITH top_ingr.ingredient AS ingredient,
     count(r)            AS recipe_count,
     avg(top_ingr.co2)   AS avg_co2e_kg
RETURN ingredient, recipe_count, round(avg_co2e_kg, 4) AS avg_co2e_kg
ORDER BY avg_co2e_kg DESC
LIMIT 20
"""
rq1a_df = pd.DataFrame(run_query(rq1a_q))
print("RQ1a — Ingredients most often responsible for the highest CO₂ in a recipe:")
display(rq1a_df)
rq1a_df.to_csv("rq_exploration_martin/rq1a_dominant_ingredient_co2.csv", index=False)
print("Saved: rq_exploration_martin/rq1a_dominant_ingredient_co2.csv")


RQ1a — Ingredients most often responsible for the highest CO₂ in a recipe:


,ingredient,recipe_count,avg_co2e_kg
0,beef,1098,29.9139
1,mutton,1003,19.6728
2,shrimp,938,12.4408
3,paneer,1650,10.6248
4,chicken,725,6.3761
5,fish,723,5.6419
6,butter,542,4.2909
7,ghee,531,4.1978
8,cream,295,1.7255
9,basmati rice,335,1.5277


Saved: rq_exploration_martin/rq1a_dominant_ingredient_co2.csv


In [6]:
## RQ1b — Ingredient frequency and avg CO₂ in top-1000 highest-rated recipes
rq1b_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL AND r.avg_rating IS NOT NULL
WITH r ORDER BY r.avg_rating DESC LIMIT 1000
WITH collect(DISTINCT r) AS top_recipes
UNWIND top_recipes AS r
MATCH (r)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
RETURN i.ingredient_name                AS ingredient,
       count(r)                         AS recipe_count,
       round(avg(h.co2e_kg), 4)         AS avg_co2e_kg,
       round(sum(h.co2e_kg), 4)         AS total_co2e_kg
ORDER BY avg_co2e_kg DESC
LIMIT 20
"""
rq1b_df = pd.DataFrame(run_query(rq1b_q))
print("RQ1b — Ingredient CO₂ profile in top-1000 rated recipes:")
display(rq1b_df)
rq1b_df.to_csv("rq_exploration_martin/rq1b_top_rated_ingredient_co2.csv", index=False)
print("Saved: rq_exploration_martin/rq1b_top_rated_ingredient_co2.csv")


RQ1b — Ingredient CO₂ profile in top-1000 rated recipes:


,ingredient,recipe_count,avg_co2e_kg,total_co2e_kg
0,beef,19,29.5073,560.6389
1,mutton,12,10.9050,130.8600
2,shrimp,21,7.4022,155.4470
3,paneer,23,7.2782,167.3988
4,chicken,13,4.6640,60.6314
5,ghee,7,3.4944,24.4610
6,butter,15,3.3865,50.7976
7,fish,17,3.2199,54.7388
8,cream,11,1.5458,17.0034
9,basmati rice,11,1.2859,14.1450


Saved: rq_exploration_martin/rq1b_top_rated_ingredient_co2.csv


## RQ2 — Which ingredient pairs co-occur most frequently in high-carbon recipes?

Take the top-500 recipes by total CO₂ footprint and find pairs of ingredients that appear together most often.


In [7]:
## RQ2 — Co-occurring ingredient pairs in top-500 high-carbon recipes
rq2_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
WITH r, sum(h.co2e_kg) AS total_co2e
ORDER BY total_co2e DESC LIMIT 500
WITH collect(r) AS top_recipes
UNWIND top_recipes AS r
MATCH (r)-[:HAS_INGREDIENT]->(i1:Ingredient)
MATCH (r)-[:HAS_INGREDIENT]->(i2:Ingredient)
WHERE i1.ingredient_name < i2.ingredient_name
RETURN i1.ingredient_name AS ingredient_1,
       i2.ingredient_name AS ingredient_2,
       count(r)           AS co_occurrence
ORDER BY co_occurrence DESC
LIMIT 30
"""
rq2_df = pd.DataFrame(run_query(rq2_q))
print("RQ2 — Top ingredient co-occurrence pairs in 500 highest-carbon recipes:")
display(rq2_df)
rq2_df.to_csv("rq_exploration_martin/rq2_ingredient_cooccurrence.csv", index=False)
print("Saved: rq_exploration_martin/rq2_ingredient_cooccurrence.csv")


RQ2 — Top ingredient co-occurrence pairs in 500 highest-carbon recipes:


,ingredient_1,ingredient_2,co_occurrence
0,onion,salt,500
1,garlic,salt,500
2,garlic,onion,500
3,beef,onion,456
4,beef,salt,456
5,beef,garlic,456
6,garlic,paneer,135
7,onion,paneer,135
8,paneer,salt,135
9,beef,paneer,106


Saved: rq_exploration_martin/rq2_ingredient_cooccurrence.csv


In [8]:
## RQ2b — Avg pair CO₂, filtered to pairs where at least one ingredient contributes ≥ 1.0 kg CO₂e
rq2b_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
WITH r, sum(h.co2e_kg) AS total_co2e
ORDER BY total_co2e DESC LIMIT 500
WITH collect(r) AS top_recipes
UNWIND top_recipes AS r
MATCH (r)-[h1:HAS_INGREDIENT]->(i1:Ingredient)
MATCH (r)-[h2:HAS_INGREDIENT]->(i2:Ingredient)
WHERE i1.ingredient_name < i2.ingredient_name
  AND (h1.co2e_kg > 1.0 OR h2.co2e_kg > 1.0)
RETURN i1.ingredient_name                                   AS ingredient_1,
       i2.ingredient_name                                   AS ingredient_2,
       count(r)                                             AS co_occurrence,
       round(avg(h1.co2e_kg + h2.co2e_kg), 4)              AS avg_pair_co2e_kg
ORDER BY co_occurrence DESC
LIMIT 30
"""
rq2b_df = pd.DataFrame(run_query(rq2b_q))
print("RQ2b — Co-occurrence pairs with ≥1 high-carbon ingredient (top-500 carbon recipes):")
display(rq2b_df)
rq2b_df.to_csv("rq_exploration_martin/rq2_ingredient_cooccurrence_v2.csv", index=False)
print("Saved: rq_exploration_martin/rq2_ingredient_cooccurrence_v2.csv")


RQ2b — Co-occurrence pairs with ≥1 high-carbon ingredient (top-500 carbon recipes):


,ingredient_1,ingredient_2,co_occurrence,avg_pair_co2e_kg
0,beef,onion,455,48.2370
1,beef,garlic,455,48.1596
2,beef,salt,455,48.1384
3,paneer,salt,115,6.9184
4,onion,paneer,115,7.0109
5,garlic,paneer,115,6.9369
6,beef,paneer,106,50.6563
7,beef,fenugreek leaves,85,46.7825
8,garlic,ghee,80,3.9746
9,ghee,onion,80,4.0583


Saved: rq_exploration_martin/rq2_ingredient_cooccurrence_v2.csv


## RQ3 — Nutritional profile vs carbon footprint vs user engagement

Computes recipe-level CO₂ from the graph, joins with nutritional and engagement properties, then:
- Exports the full dataset for statistical analysis
- Prints a correlation matrix
- Groups recipes into high/low carbon and compares means
- Runs Mann-Whitney U tests (non-parametric, appropriate for right-skewed CO₂ distribution)


In [9]:
## RQ3 — Recipe-level CO₂ + nutritional + engagement dataset from graph
rq3_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(:Ingredient)
WHERE h.co2e_kg IS NOT NULL
WITH r, sum(h.co2e_kg) AS total_co2e_kg
RETURN r.recipe_id           AS recipe_id,
       r.name                AS recipe_name,
       r.cuisine             AS cuisine,
       r.carbon_tier         AS carbon_tier,
       total_co2e_kg,
       r.calories            AS calories,
       r.protein_g           AS protein_g,
       r.fat_g               AS fat_g,
       r.sugar_g             AS sugar_g,
       r.fiber_g             AS fiber_g,
       r.carbohydrates_g     AS carbohydrates_g,
       r.sodium_mg           AS sodium_mg,
       r.avg_rating          AS avg_rating,
       r.total_reviews       AS total_reviews,
       r.is_vegetarian       AS is_vegetarian,
       r.is_vegan            AS is_vegan,
       r.servings            AS servings,
       r.estimated_cost_usd  AS estimated_cost_usd
"""
rq3_df = pd.DataFrame(run_query(rq3_q))
print(f"RQ3 dataset: {len(rq3_df):,} recipes with CO₂ data")
rq3_df.to_csv("rq_exploration_martin/rq3_recipe_co2_nutrition.csv", index=False)
print("Saved: rq_exploration_martin/rq3_recipe_co2_nutrition.csv")
display(rq3_df.describe())


RQ3 dataset: 9,997 recipes with CO₂ data
Saved: rq_exploration_martin/rq3_recipe_co2_nutrition.csv


,total_co2e_kg,calories,protein_g,fat_g,sugar_g,fiber_g,carbohydrates_g,sodium_mg,avg_rating,total_reviews,servings,estimated_cost_usd
count,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000,9997.000000
mean,12.353558,477.057617,26.195659,24.240972,10.514654,7.980394,54.537761,851.229669,4.249765,2499.691607,4.567770,13.570204
std,12.827896,186.485590,13.195225,7.579553,5.142868,3.783694,14.406440,374.132705,0.434312,1444.488533,1.694722,6.677949
min,0.109797,150.000000,5.000000,10.000000,2.000000,2.000000,30.000000,200.000000,3.500000,10.000000,2.000000,2.000000
25%,3.076082,316.000000,15.000000,18.000000,6.000000,5.000000,42.000000,531.000000,3.900000,1250.000000,3.000000,7.890000
50%,7.925266,481.000000,26.000000,24.000000,11.000000,8.000000,54.000000,851.000000,4.300000,2495.000000,5.000000,13.540000
75%,17.235915,637.000000,38.000000,29.000000,15.000000,11.000000,67.000000,1173.000000,4.600000,3745.000000,6.000000,19.480000
max,72.047897,799.000000,49.000000,39.000000,19.000000,14.000000,79.000000,1499.000000,5.000000,4999.000000,7.000000,25.000000


In [10]:
## RQ3 — Correlation matrix: CO₂ vs nutritional features vs engagement
numeric_cols = [
    "total_co2e_kg", "calories", "protein_g", "fat_g", "sugar_g",
    "fiber_g", "carbohydrates_g", "sodium_mg", "avg_rating",
    "total_reviews", "servings", "estimated_cost_usd",
]
corr = rq3_df[numeric_cols].apply(pd.to_numeric, errors="coerce").corr()
print("Correlation matrix (all numeric features):")
display(corr.round(3))
corr.to_csv("rq_exploration_martin/rq3_correlation_matrix.csv")
print("Saved: rq_exploration_martin/rq3_correlation_matrix.csv")


Correlation matrix (all numeric features):


,total_co2e_kg,calories,protein_g,fat_g,sugar_g,fiber_g,carbohydrates_g,sodium_mg,avg_rating,total_reviews,servings,estimated_cost_usd
total_co2e_kg,1.000,0.006,0.398,0.234,0.006,0.006,-0.016,0.013,0.008,0.001,0.007,-0.001
calories,0.006,1.000,-0.012,-0.001,-0.007,0.012,-0.001,-0.006,-0.001,0.008,0.016,-0.002
protein_g,0.398,-0.012,1.000,0.396,0.005,0.002,-0.009,0.028,0.004,-0.003,-0.004,-0.005
fat_g,0.234,-0.001,0.396,1.000,-0.010,-0.013,-0.012,0.004,0.001,0.002,-0.007,0.006
sugar_g,0.006,-0.007,0.005,-0.010,1.000,0.002,-0.005,0.021,-0.011,-0.012,0.017,0.001
fiber_g,0.006,0.012,0.002,-0.013,0.002,1.000,0.008,-0.006,0.014,0.001,-0.006,-0.011
carbohydrates_g,-0.016,-0.001,-0.009,-0.012,-0.005,0.008,1.000,0.006,-0.016,-0.008,-0.012,-0.014
sodium_mg,0.013,-0.006,0.028,0.004,0.021,-0.006,0.006,1.000,0.013,0.006,0.006,-0.000
avg_rating,0.008,-0.001,0.004,0.001,-0.011,0.014,-0.016,0.013,1.000,-0.017,0.001,0.010
total_reviews,0.001,0.008,-0.003,0.002,-0.012,0.001,-0.008,0.006,-0.017,1.000,0.016,0.004


Saved: rq_exploration_martin/rq3_correlation_matrix.csv


In [11]:
## RQ3 — High vs low carbon group comparison (split at median)
df_num = rq3_df.copy()
df_num["total_co2e_kg"] = pd.to_numeric(df_num["total_co2e_kg"], errors="coerce")

median_co2 = df_num["total_co2e_kg"].median()
df_num["carbon_group"] = df_num["total_co2e_kg"].apply(
    lambda x: "high" if x >= median_co2 else "low"
)

compare_cols = [
    "calories", "protein_g", "fat_g", "sugar_g", "fiber_g",
    "carbohydrates_g", "avg_rating", "total_reviews", "estimated_cost_usd",
]
for c in compare_cols:
    df_num[c] = pd.to_numeric(df_num[c], errors="coerce")

group_means = df_num.groupby("carbon_group")[compare_cols].mean().round(3)
print(f"Median CO₂ split: {median_co2:.4f} kg CO₂eq\n")
print("Mean nutritional + engagement metrics by carbon group:")
display(group_means)
group_means.to_csv("rq_exploration_martin/rq3_high_low_carbon_comparison.csv")
print("Saved: rq_exploration_martin/rq3_high_low_carbon_comparison.csv")


Median CO₂ split: 7.9253 kg CO₂eq

Mean nutritional + engagement metrics by carbon group:


,calories,protein_g,fat_g,sugar_g,fiber_g,carbohydrates_g,avg_rating,total_reviews,estimated_cost_usd
carbon_group,,,,,,,,,
high,478.496,32.07,26.199,10.521,7.993,54.376,4.253,2496.853,13.597
low,475.619,20.32,22.283,10.508,7.967,54.699,4.246,2502.530,13.543


Saved: rq_exploration_martin/rq3_high_low_carbon_comparison.csv


In [12]:
## RQ3 — Mann-Whitney U tests: high vs low carbon group for each nutritional/engagement variable
# Mann-Whitney U is used instead of t-test because the CO₂ distribution is right-skewed.
from scipy import stats

test_cols = ["calories", "protein_g", "fat_g", "sugar_g",
             "fiber_g", "carbohydrates_g", "avg_rating", "total_reviews"]

mw_results = []
for col in test_cols:
    high_vals = df_num[df_num["carbon_group"] == "high"][col].dropna()
    low_vals  = df_num[df_num["carbon_group"] == "low"][col].dropna()
    stat, p = stats.mannwhitneyu(high_vals, low_vals, alternative="two-sided")
    mw_results.append({
        "variable":           col,
        "U_statistic":        round(stat, 2),
        "p_value":            round(p, 4),
        "significant_at_0.05": p < 0.05,
    })

mw_df = pd.DataFrame(mw_results)
print("Mann-Whitney U test results (high CO₂ vs low CO₂ recipes):")
display(mw_df)
mw_df.to_csv("rq_exploration_martin/rq3_mannwhitney_results.csv", index=False)
print("Saved: rq_exploration_martin/rq3_mannwhitney_results.csv")

sig     = mw_df[mw_df["significant_at_0.05"]]
not_sig = mw_df[~mw_df["significant_at_0.05"]]
print(f"\n  Significant (p < 0.05):      {sig['variable'].tolist()}")
print(f"  Not significant (p >= 0.05): {not_sig['variable'].tolist()}")


Mann-Whitney U test results (high CO₂ vs low CO₂ recipes):


,variable,U_statistic,p_value,significant_at_0.05
0,calories,12599404.5,0.4587,False
1,protein_g,18992912.0,0.0000,True
2,fat_g,16092863.5,0.0000,True
3,sugar_g,12511535.5,0.8949,False
4,fiber_g,12541469.5,0.7335,False
5,carbohydrates_g,12330069.5,0.2602,False
6,avg_rating,12606356.5,0.4291,False
7,total_reviews,12464112.5,0.8440,False


Saved: rq_exploration_martin/rq3_mannwhitney_results.csv

  Significant (p < 0.05):      ['protein_g', 'fat_g']
  Not significant (p >= 0.05): ['calories', 'sugar_g', 'fiber_g', 'carbohydrates_g', 'avg_rating', 'total_reviews']


## Schema Alignment — carbon_tier Distribution

Uses the `carbon_tier` derived property set by inference rule R1 during ingestion (`ingest_desktop.py`).  
Tiers: `low` (< 2 kg CO₂eq) · `medium` (2–8 kg) · `high` (> 8 kg)


In [13]:
## carbon_tier distribution across all recipes
tier_q = """
MATCH (r:Recipe)
WHERE r.carbon_tier IS NOT NULL
RETURN r.carbon_tier AS tier, count(r) AS recipe_count
ORDER BY recipe_count DESC
"""
tier_df = pd.DataFrame(run_query(tier_q))
print("carbon_tier distribution:")
display(tier_df)


carbon_tier distribution:


,tier,recipe_count
0,high,4969
1,medium,3472
2,low,1556


In [14]:
## Schema verification — confirm graph labels and relationship types match ontology
label_q = "CALL db.labels() YIELD label RETURN label ORDER BY label"
rel_q   = "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType"

labels = [r["label"]            for r in run_query(label_q)]
rels   = [r["relationshipType"] for r in run_query(rel_q)]

print("=== Node labels in graph ===")
for lbl in labels:
    print(f"  {lbl}")

print("\n=== Relationship types in graph ===")
for rel in rels:
    print(f"  {rel}")

print("\n=== Expected ontology schema ===")
expected_labels = ["Cuisine", "Ingredient", "NutritionalProfile", "Recipe"]
expected_rels   = ["BELONGS_TO_CUISINE", "HAS_INGREDIENT", "HAS_NUTRITION"]

missing_labels = [l for l in expected_labels if l not in labels]
missing_rels   = [r for r in expected_rels   if r not in rels]

if missing_labels:
    for l in missing_labels:
        print(f"  MISSING label:        {l}")
if missing_rels:
    for r in missing_rels:
        print(f"  MISSING relationship: {r}")

if not missing_labels and not missing_rels:
    print("  Graph matches full ontology schema. ✓")


=== Node labels in graph ===
  Cuisine
  Ingredient
  NutritionalProfile
  Recipe

=== Relationship types in graph ===
  BELONGS_TO_CUISINE
  HAS_INGREDIENT
  HAS_NUTRITION

=== Expected ontology schema ===
  Graph matches full ontology schema. ✓


In [15]:
## Close driver
driver.close()
print("Driver closed.")


Driver closed.
